In [1]:
import torch
import pandas as pd

df = pd.read_csv('/kaggle/input/english-french/fra.txt', sep='\t', header=None,
    usecols=[0, 1],
    names=['english', 'french'])

In [2]:
# Dependency run only when runtime was changed
!pip install ftfy
!pip install contractions
!pip install swifter
!pip install gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 6.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.9/113.9 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.2 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16505 sha256=b87f36d5c54f68642232210771abc97e06ffd3ce80de121f09828ed8f5ab4d44
  Stored in directory: /root/.cache/pip/wheels/ef/7f/bd/9bed48f078f3ee1fa75e0b29b6e0335ce1cb03a38d3443b3a3
Successfully built swifter


## File downloading

In [3]:
!gdown --id 1xtqo0GPC2kJbYYEZBgcG-nU-3l70hLL1 -O /kaggle/working/encoder_decoder.pth

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1xtqo0GPC2kJbYYEZBgcG-nU-3l70hLL1
From (redirected): https://drive.google.com/uc?id=1xtqo0GPC2kJbYYEZBgcG-nU-3l70hLL1&confirm=t&uuid=4d727c58-2ef0-431e-abd0-537e42ac3fc7
To: /kaggle/working/encoder_decoder.pth
100%|████████████████████████████████████████| 712M/712M [00:11<00:00, 62.7MB/s]


## **preprocessing**

In [4]:
import re
import html
import unicodedata
import ftfy
import contractions

def clean_and_restore_text(text, lang='en', use_ftfy=True):

    text = html.unescape(text)
    text = text.encode('utf-8').decode('unicode_escape', errors='ignore')

    if use_ftfy:
        text = ftfy.fix_text(text)

    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()

    text = re.sub(r"[“”«»„‟]", '"', text)
    text = re.sub(r"[‘’‚‛]", "'", text)
    text = text.replace("\\'", "'").replace('\\"', '"')
    text = re.sub(r'\s+([?.!,;:])', r'\1', text)
    text = re.sub(r'([?.!,;:])(?=[^\s])', r'\1 ', text)

    if lang == 'en':
        text = contractions.fix(text)


    if text and text[0].islower():
        text = text[0].upper() + text[1:]

    return text

def restore_sentence_formatting(text, lang='en'):
    """
    Fixes spacing, quotes, and punctuation issues in cleaned text.
    Works for English and French text.
    """

    text = re.sub(r'"\s+', '"', text)
    text = re.sub(r'\s+"', '"', text)

    if lang == 'fr':

        text = re.sub(r'\s+([.,])', r'\1', text)
        text = re.sub(r'([?!:;])', r' \1', text)
        text = re.sub(r'\s+', ' ', text).strip()
    else:
        text = re.sub(r'\s+([?.!,;:])', r'\1', text)
        text = re.sub(r'([?.!,;:])(?=[^\s"])', r'\1 ', text)


    text = re.sub(r'\s+', ' ', text).strip()


    def capitalize_after_period(match):
        return match.group(1) + match.group(2).upper()
    text = re.sub(r'([.!?]\s+)([a-z])', capitalize_after_period, text)

    return text

# Only for english
def reversing(sentence):
  words = sentence.split(' ')
  reversed_words = words[::-1]
  reversed_sentence = ' '.join(reversed_words)
  return reversed_sentence


import swifter
df['english'] = df['english'].swifter.apply(lambda x: clean_and_restore_text(x, lang='en'))
df['french'] = df['french'].swifter.apply(lambda x: clean_and_restore_text(x, lang='fr'))

df['english'] = df['english'].apply(lambda x: restore_sentence_formatting(x, lang='en'))
df['french'] = df['french'].apply(lambda x: restore_sentence_formatting(x, lang='fr'))

df['english'] = df['english'].apply(lambda x: reversing(x))

Pandas Apply:   0%|          | 0/239189 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/239189 [00:00<?, ?it/s]

## **Tokenizing**

In [5]:
def word_tokinize(text):
    PUNCTUATION_TO_STRIP = '.,!?"()[]{}<>:;/=_+*&^%$#@`~'

    dirty_tokens = text.lower().split()
    clean_tokens = []

    for token in dirty_tokens:
        clean_token = token.strip(PUNCTUATION_TO_STRIP)
        if clean_token:
            clean_tokens.append(clean_token)

    return clean_tokens

def forming_vocab(text, Lang):
  tokinize_text = word_tokinize(text)
  lang = Lang

  if lang == 'eng':
    for word in tokinize_text:
      if word not in english_vocab:
        english_vocab[word] = len(english_vocab)

  if lang == 'fre':
    for word in tokinize_text:
      if word not in french_vocab:
        french_vocab[word] = len(french_vocab)

english_vocab = {'<pad>':0, '<unk>':1}
french_vocab = {'<pad>':0, '<unk>':1, '<start>':2, '<end>':3}

df['english'].apply(lambda text: forming_vocab(text, 'eng'))
df['french'].apply(lambda text: forming_vocab(text, 'fre'))

print("English vocab:",len(english_vocab))
print("French vocab:",len(french_vocab))

English vocab: 18003
French vocab: 34020


## **Text To Sequence**

In [6]:
import numpy as np

def text_to_numbers(sentence, vocab, language):
  sentence_tokinize = word_tokinize(sentence)
  vocab = vocab
  lang = language
  text_seq =[]

  if lang == 'eng':
    for word in sentence_tokinize:
      if word in vocab:
        text_seq.append(vocab[word])
      else:
        text_seq.append(vocab['<unk>'])

  if lang == 'fre':
    text_seq.append(1)

    for word in sentence_tokinize:
      if word in vocab:
        text_seq.append(vocab[word])
      else:
        text_seq.append(vocab['<unk>'])
    text_seq.append(2)

  return text_seq

english_sequences = df['english'].apply(lambda sentence: text_to_numbers(sentence, english_vocab, 'eng'))
french_sequences = df['french'].apply(lambda sentence: text_to_numbers(sentence, french_vocab, 'fre'))

## **Padding-Data_preparing**

In [7]:
# English have 54 highest len
# French have 57 hightest len

def padding(max_len, sequence, mod):
  if mod == 'start':
    List = []
    zeroes = [0] * (max_len - len(sequence))
    List = zeroes + sequence

  elif mod =='end':
    List = []
    zeroes = [0] * (max_len - len(sequence))
    List = sequence + zeroes

  return List

from torch.utils.data import Dataset, DataLoader

class data_formation(Dataset):
  def __init__(self, english_sequences, french_sequences):
    self.english_sequences = english_sequences
    self.french_sequences = french_sequences

  def __len__(self):
    return self.english_sequences.shape[0]

  def __getitem__(self, index):
    english_sequence = padding(54, self.english_sequences[index], mod='start')
    french_sequence = padding(57, self.french_sequences[index], mod='end')

    return torch.tensor(english_sequence, dtype= torch.long), torch.tensor(french_sequence, dtype=torch.long)

# Data Spliting
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(english_sequences.values, french_sequences.values, test_size=0.1, shuffle=True, random_state=42)

training_dataset = data_formation(X_train, y_train)
testing_dataset = data_formation(X_test, y_test)

import os
training_loader = DataLoader(training_dataset, batch_size=224, shuffle= True, num_workers=4, pin_memory=True)
testing_loader = DataLoader(testing_dataset, batch_size=224, pin_memory=True)

## **Encoder-Decoder**

In [8]:
import torch.nn as nn

class encoder(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 512, padding_idx=0)
    self.lstm = nn.LSTM(512,512, 3, batch_first=True, dropout=0.2)

  def forward(self,X):
    embedding_output = self.embedding(X)
    all_hidden_state, (ht, ct) = self.lstm(embedding_output)

    return all_hidden_state, ht, ct


class decoder(nn.Module):
  def __init__(self, vocab_size, enc_hidden_size, dec_hidden_size):
    super().__init__()
    self.attention = attention(enc_hidden_size, dec_hidden_size)

    self.embedding = nn.Embedding(vocab_size, 512, padding_idx=0)
    self.lstm = nn.LSTM(512 + enc_hidden_size, 512, 3, batch_first=True, dropout=0.2)
    self.network = nn.Linear(512,vocab_size)

  def forward(self,decoder_input, decoder_p_state, encoder_ht, encoder_ct, encoder_hiddens):
    embedding_output = self.embedding(decoder_input)
    context_vector = self.attention(encoder_hiddens,decoder_p_state)

    lstm_input = torch.cat((embedding_output,context_vector), dim=2)

    all_hidden_state, (ht, ct) = self.lstm(lstm_input, (encoder_ht, encoder_ct))

    network_output = self.network(all_hidden_state)
    network_output = network_output.squeeze(1)

    return network_output, ht, ct


class attention(nn.Module):
  def __init__(self, enc_hidden_size, dec_hidden_size):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(enc_hidden_size + dec_hidden_size,1024),
        nn.Tanh(),
        nn.Linear(1024,512),
        nn.Tanh(),
        nn.Linear(512,1)
    )
  def forward(self, encoder_hiddens, decoder_p_ht):
    # encoder and decoder are not same, also with this decoder is a single vector to calculate weghts we need keep them equal for easy calculation

    decoder_hidden_expanded = decoder_p_ht.unsqueeze(1).repeat(1, encoder_hiddens.shape[1], 1)
    input = torch.cat((encoder_hiddens, decoder_hidden_expanded), dim=2) # here we are concat the encoder_hts and decoder_hts

    weights = self.network(input)
    weights = weights.squeeze(2)
    attn_weights = torch.softmax(weights, dim=1)

    context_vector = torch.bmm(attn_weights.unsqueeze(1), encoder_hiddens)

    return context_vector


class seq2seq(nn.Module):
  def __init__(self, encoder, decoder):
    super().__init__()
    self.encoder = encoder
    self.decoder = decoder

  def forward(self, encoder_input, decoder_input):
    encoder_hiddens, encoder_ht, encoder_ct = self.encoder(encoder_input)
    current_ht = encoder_ht
    current_ct = encoder_ct
    decoder_p_state = current_ht[-1]
    predictions = torch.zeros(decoder_input.shape[0],decoder_input.shape[1]-1,self.decoder.network.out_features, device=encoder_input.device)

    input_token = decoder_input[:, 0].unsqueeze(1)

    for t in range(decoder_input.shape[1]-1):

      decoder_pred, next_ht, next_ct = self.decoder(
          input_token,
          decoder_p_state=decoder_p_state,
          encoder_ht = current_ht,
          encoder_ct= current_ct,
          encoder_hiddens= encoder_hiddens
      )
      #next_ht, next_ct = next_states
      decoder_p_state = next_ht[-1]
      current_ht = next_ht
      current_ct = next_ct

      input_token = decoder_input[:, t+1].unsqueeze(1)
      predictions[:, t, :] = decoder_pred

    return predictions

## **Training & testing Function**

In [9]:
from torch.cuda.amp import GradScaler, autocast
scaler = GradScaler()

def train(model, data_loader, loss_function, optimizer, device, total_epochs):

  model.train()
  total_loss = 0

  for X, y in data_loader:
    X, y = X.to(device), y.to(device)

    decoder_input = y
    decoder_target = y[:, 1:]

    optimizer.zero_grad()

    with autocast():
      # 1.Forward pass
      pred = model(X,decoder_input)

    # 2. Reshape for loss function
    pred_for_loss = pred.view(-1, pred.shape[2])
    y_for_loss = decoder_target.reshape(-1)

    # 3. Calculate loss
    loss = loss_function(pred_for_loss, y_for_loss)

    # 4. Backpropagation

    scaler.scale(loss).backward() # loss.backwar()
    scaler.step(optimizer) #optimizer.step()

    scaler.update()


    total_loss += loss.item()

  return total_loss / len(data_loader)

def test(model, data_loader, loss_function, device, total_epochs):

  model.eval()
  total_loss = 0

  with torch.no_grad():
    for X, y in data_loader:
      X, y = X.to(device), y.to(device)

      decoder_input = y
      decoder_target = y[:, 1:]

      # 1. Forward pass
      pred = model(X, decoder_input)

      # 2. Reshape
      pred_for_loss = pred.view(-1, pred.shape[2])
      y_for_loss = decoder_target.reshape(-1)

      # 3. Calculate loss
      loss = loss_function(pred_for_loss, y_for_loss)
      total_loss += loss.item()

  return total_loss / len(data_loader)

## **Training_process**

In [12]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
import os 
enc = encoder(len(english_vocab)+1)
dec = decoder(len(french_vocab)+1, 512, 512)
model = seq2seq(enc, dec)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params}")

# hyper parameters
epochs = 35
learning_rate = 1e-3
loss_function = torch.nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate, weight_decay=0.0001)

scheduler = ReduceLROnPlateau(optimizer, 'min', patience=2,factor=0.5)

CHECKPOINT_FILE = "encoder_decoder.pth"

if os.path.exists(CHECKPOINT_FILE):
    print("Loading checkpoint...")
    checkpoint = torch.load(CHECKPOINT_FILE)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1 # Start from the NEXT epoch
    history = checkpoint['history']

    print(f"Resuming training from epoch {start_epoch}")
else:
    print("Starting new training...")
    start_epoch = 0
    history = {'train_loss': [], 'val_loss': []}
    
for epoch in range(start_epoch, epochs):
    
    avg_train_loss = train(model, training_loader, loss_function, optimizer, device, epochs)
    avg_val_loss = test(model, testing_loader, loss_function, device, epochs)

    scheduler.step(avg_val_loss)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)

    print(f"Epoch: {epoch + 1}/{epochs}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss:   {avg_val_loss:.4f}")
    
    print("Saving checkpoint...")
    checkpoint = {
        'epoch': epoch, # The epoch we just finished
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history
      }
    torch.save(checkpoint, CHECKPOINT_FILE)
print("Training finished!")

Number of trainable parameters: 59320550
Loading checkpoint...
Resuming training from epoch 30
Epoch: 31/35
Train Loss: 1.5336
Val Loss:   1.6451
Saving checkpoint...
Epoch: 32/35
Train Loss: 1.5275
Val Loss:   1.8324
Saving checkpoint...
Epoch: 33/35
Train Loss: 1.5292
Val Loss:   1.7006
Saving checkpoint...
Epoch: 34/35
Train Loss: 1.5246
Val Loss:   1.6582
Saving checkpoint...
Epoch: 35/35
Train Loss: 1.3956
Val Loss:   1.5557
Saving checkpoint...
Training finished!


In [11]:
from IPython.display import FileLink
FileLink('encoder_decoder.pth')

/kaggle/working/encoder_decoder.pth